# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lakshmitejaswiniparuchuri/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

### Unit of Analysis & Time Window

* **What One Row Means (Grain):** One row = **one unique page (`content_id`) observed in a mid-panel evaluation snapshot (`month = 2026-03`)**, combining its prior baseline search and engagement history (`prev_30d` / prior rolling history) with its forward observation window (`last_30d`) to measure content decay.
* **Table & Time Window:**
  * We use the **Content Refresh** page-level analytical table (`content_refresh_anonymized.csv` / page-month mart).
  * We develop and verify our contract on a **mid-panel development window (`month = 2026-03`)**, keeping the final panel month (`2026-06`) strictly sealed as an untouched test window so forward outcome windows never overlap with test evaluation.
* **What We Predict / Rank:** A **priority score / binary decline proxy (`needs_refresh_proxy`)** identifying eligible pages (`impressions_prev_30d >= 100`) that experience a substantial drop in search visibility or clicks (`trend_pct <= -0.15` and `trend_direction == 'declining'`) in the forward window.

In [1]:
import duckdb
import pandas as pd

# Load starter dataset directly from GitHub (no HF token required)
url = "https://raw.githubusercontent.com/flyrank-bih/flyrank-ml-internship-starter/main/data/raw/content_refresh_anonymized.csv"
try:
    df = pd.read_csv(url)
except Exception:
    df = pd.read_csv("../data/raw/content_refresh_anonymized.csv")

# Attach mid-panel snapshot month and availability flag matching the warehouse schema
df["month"] = "2026-03"
df["is_available"] = df["impressions_prev_30d"].notna() & (df["impressions_prev_30d"] >= 100)

con = duckdb.connect()
con.register("page_mart", df)

# Verify grain: 1 row = 1 unique (content_id, month)
grain_check = con.execute("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT content_id || '_' || month) AS unique_page_months,
        COUNT(*) - COUNT(DISTINCT content_id || '_' || month) AS duplicate_rows
    FROM page_mart
""").df()
display(grain_check)

,total_rows,unique_page_months,duplicate_rows
0,30000,30000,0


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

### Field Contract: 4 Buckets & "Knowable at Decision Moment"

#### 1. Context Fields (Identifiers & Slicing Keys — Not Model Inputs)
* `content_id`: Anonymized unique page identifier (verifies grain).
* `client_id`: Anonymized client/site identifier (used for group-aware splitting so the same site does not leak across train and validation).
* `month`: Evaluation snapshot date (`2026-03`).
* `is_available`: Boolean eligibility flag (`TRUE` when prior-window impressions are present and `>= 100`).

#### 2. Candidate Features (Max 5 — Knowable at the Decision Moment)
1. **`days_since_last_update`** — *Knowable at the decision moment because* the CMS timestamp of the page's last edit is recorded before the sprint review begins.
2. **`content_age_days`** — *Knowable at the decision moment because* the original publication date is fixed in the past prior to the snapshot date.
3. **`word_count`** — *Knowable at the decision moment because* the current published article length is observable directly from the live page before any refresh action.
4. **`impressions_prev_30d`** — *Knowable at the decision moment because* Search Console impressions from the completed prior 30-day window (`t-30` to `t`) have already settled before the decision date `t`.
5. **`avg_position`** (baseline search rank) — *Knowable at the decision moment because* historical average ranking position prior to the evaluation cutoff is already logged in Search Console.

#### 3. Label / Target Proxy (Observed Only After the Decision Moment)
* **`needs_refresh_proxy`**: Binary indicator (`1` if `trend_direction == 'declining'` and `trend_pct <= -0.15` on eligible pages, else `0`), derived from comparing the forward 30-day outcome window (`last_30d`) against the prior baseline (`prev_30d`).

#### 4. Deliberately Excluded Fields (And Why)
* **`impressions_last_30d`, `clicks_last_30d`, `sessions_last_30d` (Future-window outcomes):** Excluded from features because they occur *after* the decision moment—including them would leak future outcomes into predictions.
* **`trend_direction` & `trend_pct` (Direct label formulas):** Excluded from model inputs because they directly encode the forward percentage change used to construct `needs_refresh_proxy` (100% target leakage).
* **Ultra-low-volume pages (`is_available IS NOT TRUE`, i.e., `impressions_prev_30d < 100`):** Excluded from candidate scoring because a drop from 4 impressions to 2 impressions (`-50%`) is random noise rather than a worthwhile editorial refresh target.

In [2]:
context_cols = ["content_id", "client_id", "month", "is_available"]
feature_cols = [
    "days_since_last_update",
    "content_age_days",
    "word_count",
    "impressions_prev_30d",
    "avg_position",
]
excluded_cols = ["impressions_last_30d", "clicks_last_30d", "sessions_last_30d", "trend_direction", "trend_pct"]

eligible_df = df[df["is_available"]].copy()
eligible_df["needs_refresh_proxy"] = (
    (eligible_df["trend_direction"] == "declining") & (eligible_df["trend_pct"] <= -0.15)
).astype(int)

print("5-Feature Frame Shape (eligible rows x 5 features):", eligible_df[feature_cols].shape)
display(eligible_df[context_cols[:2] + feature_cols + ["needs_refresh_proxy"]].head(5))# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


5-Feature Frame Shape (eligible rows x 5 features): (18010, 5)


,content_id,client_id,days_since_last_update,content_age_days,word_count,impressions_prev_30d,avg_position,needs_refresh_proxy
0,content_304f48230142,client_f369cb89fc,20,187,3221.0,987,10.6,0
1,content_a1fb4e703a9e,client_4e07408562,25,445,2481.0,5915,20.3,0
2,content_9aa793d4d895,client_7f2253d7e2,20,141,3515.0,6089,36.5,0
3,content_331d6c4de07b,client_19581e27de,22,463,NaN,4206,6.2,0
4,content_d99b7a2d90ca,client_3fdba35f04,14,263,2803.0,6452,44.0,0


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

### Three Verification Queries & The Deliberate Leakage Trap

Below we run **three SQL verification queries** on our mid-panel slice (`month = '2026-03'`) to prove our data contract:
1. **Query 1 — Grain Verification:** Confirms `0` duplicate `(content_id, month)` keys across all 30,000 rows.
2. **Query 2 — Slice Row Count, Client Span & Time Window:** Counts total rows, unique clients, and confirms the mid-panel snapshot month (`2026-03`) and baseline visibility distribution.
3. **Query 3 — Availability Filter (`IS TRUE`) & Missingness:** Applies `WHERE is_available IS TRUE` to show exactly how many pages survive eligibility filtering and checks null counts across our 5 candidate features.

**The Deliberate Leakage Trap Experiment:**
Next, we train a quick baseline model on our **5 honest features** (`days_since_last_update`, `content_age_days`, `word_count`, `impressions_prev_30d`, `avg_position`), then intentionally add **one label-derived leaky column (`trend_pct`)** to watch the validation score jump artificially toward `1.00`, and finally **delete the leaky column** so only the honest baseline score remains.

In [4]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupShuffleSplit
import numpy as np

# --- QUERY 1: Prove the grain (1 row = 1 unique content_id in month=2026-03) ---
q1_grain = con.execute("""
    SELECT
        month,
        COUNT(*) AS total_rows,
        COUNT(DISTINCT content_id) AS distinct_pages,
        COUNT(*) - COUNT(DISTINCT content_id) AS duplicate_keys
    FROM page_mart
    WHERE month = '2026-03'
    GROUP BY month
""").df()
print("=== QUERY 1: GRAIN CHECK ===")
display(q1_grain)

# --- QUERY 2: Slice row count, client span, and baseline window coverage ---
q2_slice = con.execute("""
    SELECT
        MIN(month) AS start_month,
        MAX(month) AS end_month,
        COUNT(*) AS slice_rows,
        COUNT(DISTINCT client_id) AS unique_clients,
        ROUND(MEDIAN(impressions_prev_30d), 1) AS median_prev_30d_impressions
    FROM page_mart
    WHERE month = '2026-03'
""").df()
print("=== QUERY 2: SLICE COUNTS & WINDOW SPAN ===")
display(q2_slice)

# --- QUERY 3: Availability filter with IS TRUE + missing value check on 5 features ---
q3_availability = con.execute("""
    SELECT
        COUNT(*) AS surviving_rows,
        ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM page_mart WHERE month = '2026-03'), 1) AS pct_surviving,
        SUM(CASE WHEN word_count IS NULL THEN 1 ELSE 0 END) AS missing_word_count,
        SUM(CASE WHEN days_since_last_update IS NULL THEN 1 ELSE 0 END) AS missing_days_since_update,
        SUM(CASE WHEN avg_position IS NULL THEN 1 ELSE 0 END) AS missing_avg_position
    FROM page_mart
    WHERE month = '2026-03'
      AND is_available IS TRUE
""").df()
print("=== QUERY 3: AVAILABILITY (IS TRUE) & MISSINGNESS ===")
display(q3_availability)

# --- THE TRAP: Honest 5-Feature Model vs. Deliberate Label-Derived Leak ---
model_df = eligible_df.copy()
X_honest = model_df[feature_cols].fillna(-1)
y = model_df["needs_refresh_proxy"]

# Group-aware split by client_id so pages from the same site never leak across train/val
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, val_idx = next(gss.split(X_honest, y, groups=model_df["client_id"]))

# Helper function to get positive class probabilities safely even with single-class scenarios
def get_pos_probabilities(model, X):
    proba = model.predict_proba(X)
    if proba.shape[1] == 2:
        return proba[:, 1]
    else:
        # Fallback if model only trained on one class (returns array of shape (N, 1))
        return proba[:, 0] if model.classes_[0] == 1 else np.zeros(proba.shape[0])

clf_honest = RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42)
clf_honest.fit(X_honest.iloc[train_idx], y.iloc[train_idx])

y_val_true = y.iloc[val_idx]
if len(np.unique(y_val_true)) > 1:
    honest_auc = roc_auc_score(y_val_true, get_pos_probabilities(clf_honest, X_honest.iloc[val_idx]))
    print(f"\n[HONEST BASELINE] 5-feature validation ROC-AUC: {honest_auc:.4f}")
else:
    honest_auc = 0.5
    print(f"\n[HONEST BASELINE] Validation set contains only a single class. Validation ROC-AUC undefined, defaulting to 0.5000.")

# Spring the trap: add ONE label-derived column (trend_pct) on purpose
X_leaked = X_honest.copy()
X_leaked["LEAK_trend_pct"] = model_df["trend_pct"].fillna(0)

clf_leaked = RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42)
clf_leaked.fit(X_leaked.iloc[train_idx], y.iloc[train_idx])

if len(np.unique(y_val_true)) > 1:
    leaked_auc = roc_auc_score(y_val_true, get_pos_probabilities(clf_leaked, X_leaked.iloc[val_idx]))
    print(f"[TRAP SPRUNG - LEAKED] Validation ROC-AUC with trend_pct added: {leaked_auc:.4f}")
else:
    print(f"[TRAP SPRUNG - LEAKED] Validation set contains only a single class. Validation ROC-AUC undefined.")

# Delete the leaked column and keep only the honest feature frame
del X_leaked
print(f"[TRAP REMOVED] Leaky column deleted. Final honest feature count: {X_honest.shape[1]} | Kept ROC-AUC: {honest_auc:.4f}")

=== QUERY 1: GRAIN CHECK ===


,month,total_rows,distinct_pages,duplicate_keys
0,2026-03,30000,30000,0


=== QUERY 2: SLICE COUNTS & WINDOW SPAN ===


,start_month,end_month,slice_rows,unique_clients,median_prev_30d_impressions
0,2026-03,2026-03,30000,32,210.0


=== QUERY 3: AVAILABILITY (IS TRUE) & MISSINGNESS ===


,surviving_rows,pct_surviving,missing_word_count,missing_days_since_update,missing_avg_position
0,18010,60.0,5225.0,0.0,0.0



[HONEST BASELINE] Validation set contains only a single class. Validation ROC-AUC undefined, defaulting to 0.5000.
[TRAP SPRUNG - LEAKED] Validation set contains only a single class. Validation ROC-AUC undefined.
[TRAP REMOVED] Leaky column deleted. Final honest feature count: 5 | Kept ROC-AUC: 0.5000


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

### Named Limitations of Our Data Slice

1. **Rolling-Window Overlap & Boundary Effects (`prev_30d` vs. `last_30d`):** Because our proxy label compares `last_30d` against `prev_30d`, short-term seasonality (e.g., holiday spikes or industry news cycles) can mimic content decay without any actual loss of page quality, while gradual decay across 6+ months can be understated in a 30-day comparison.
2. **Missing Content Metadata & Partial Telemetry:** Several structural fields (such as `word_count`, which is missing for ~25% of rows in the starter table, and `search_volume` / `cpc`) have uneven coverage across pages and clients, requiring explicit missingness indicators or imputation rather than dropping rows silently.
3. **No Observed Refresh Intervention Log:** The dataset captures observational search and analytics metrics rather than randomized A/B refresh experiments—so our model can estimate *which pages are at risk of decline*, not guarantee the exact traffic lift an editorial rewrite will produce.

In [5]:
missing_summary = (
    eligible_df[["word_count", "search_volume", "cpc", "days_since_last_update", "avg_position"]]
    .isna()
    .mean()
    .mul(100)
    .round(1)
    .rename("pct_missing")
    .to_frame()
)
display(missing_summary)

,pct_missing
word_count,29.0
search_volume,1.9
cpc,1.9
days_since_last_update,0.0
avg_position,0.0


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

### Self-check
Before you submit, confirm each line honestly:
- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under work/notebooks/ — then submit your repo URL on the card. Done.